In [1]:
from itertools import combinations

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw

# Split train and test

In [2]:
df = pd.read_csv("molecule.csv", sep=";")
test = df.sample(n=1, random_state=42)
train = df.drop(test.index)

with open("train.smi", "w") as f:
    for i, smi in zip(train.index, train["SMILES"]):
        f.write(f"mol{i},0,{smi.strip()}\n")

print("Test molecule:", test.index[0], test["SMILES"].values[0])

Test molecule: 27 COc1cc(C2c3cc4c(cc3OC(O)C2C)OCO4)cc(OC)c1OC


# Get frequent subgraphs

In [3]:
frag = pd.read_csv("frequent.csv")
ids = pd.read_csv("frequent.ids", sep=":", skiprows=1, header=None, names=["id", "molecules"])

df = frag.merge(ids, on="id")
df = df.drop_duplicates(subset="molecules").reset_index(drop=True)
df["id"] = df.index + 1

df.drop(columns="molecules").to_csv("frequent_dedup.csv", index=False)
df[["id", "molecules"]].to_csv("frequent_dedup.ids", sep=":", header=["id", "list"], index=False)
print(len(frag), "->", len(df))

35 -> 21


In [4]:
frequent_subgraphs = pd.read_csv("frequent_dedup.csv")
frequent_subgraphs = frequent_subgraphs.sort_values("s_abs", ascending=False)
frequent_subgraphs.head()

,id,description,nodes,edges,s_abs,s_rel,c_abs,c_rel
20,21,c1:c:c:c:c:c:1,6,6,46,100.000000,0,0.0
19,20,c1(:c:c:c:c:c:1)-C,7,7,44,95.652176,0,0.0
15,16,O(-c1:c:c:c:c:c:1)-C,8,8,30,65.217390,0,0.0
14,15,O(-c1:c:c:c:c(:c:1)-C)-C,9,9,29,63.043480,0,0.0
13,14,O(-C)-c1:c(:c:c(:c:c:1-O-C)-C)-O-C,13,13,25,54.347828,0,0.0


In [5]:
frags = list(frequent_subgraphs["description"].iloc[0:5])
mols = [Chem.MolFromSmiles(s) for s in frags]
img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(250, 250),
                           legends=[f"Subgraph {i+1}" for i in range(len(mols))])
img.save("top5.png")

# Rules

In [6]:
train["name"] = "mol" + train.index.astype(str)
train_molecule_names = set(train["name"])

In [7]:
fragment_table = pd.read_csv("frequent_dedup.csv")
fragment_molecule_lists = pd.read_csv(
    "frequent_dedup.ids", sep=":", skiprows=1, header=None,
    names=["fragment_id", "molecule_list"],
)

molecules_subgraphs_appearance = {
    (row.fragment_id,): {name.strip() for name in row.molecule_list.split(",")}
    for row in fragment_molecule_lists.itertuples()
}


for (fragment_a,), (fragment_b,) in combinations(sorted(molecules_subgraphs_appearance), 2):
    molecules_with_both = molecules_subgraphs_appearance[(fragment_a,)] & molecules_subgraphs_appearance[(fragment_b,)]
    if molecules_with_both:
        molecules_subgraphs_appearance[(fragment_a, fragment_b)] = molecules_with_both

In [8]:
molecules_subgraphs_appearance

{(1,): {'mol2', 'mol3', 'mol37', 'mol38', 'mol4', 'mol5', 'mol6'},
 (2,): {'mol32',
  'mol34',
  'mol35',
  'mol36',
  'mol38',
  'mol40',
  'mol41',
  'mol42'},
 (3,): {'mol31',
  'mol32',
  'mol33',
  'mol34',
  'mol35',
  'mol36',
  'mol37',
  'mol38',
  'mol39',
  'mol40'},
 (4,): {'mol31',
  'mol32',
  'mol33',
  'mol34',
  'mol35',
  'mol36',
  'mol37',
  'mol38',
  'mol39',
  'mol40',
  'mol41',
  'mol42'},
 (5,): {'mol0',
  'mol1',
  'mol10',
  'mol2',
  'mol3',
  'mol4',
  'mol5',
  'mol6',
  'mol7',
  'mol8',
  'mol9'},
 (6,): {'mol0',
  'mol1',
  'mol10',
  'mol11',
  'mol12',
  'mol2',
  'mol3',
  'mol4',
  'mol5',
  'mol6',
  'mol7',
  'mol8',
  'mol9'},
 (7,): {'mol10',
  'mol32',
  'mol34',
  'mol35',
  'mol36',
  'mol38',
  'mol40',
  'mol41',
  'mol42'},
 (8,): {'mol22',
  'mol31',
  'mol32',
  'mol33',
  'mol34',
  'mol35',
  'mol36',
  'mol37',
  'mol38',
  'mol39',
  'mol40'},
 (9,): {'mol0', 'mol1', 'mol21', 'mol3', 'mol4', 'mol5', 'mol6'},
 (10,): {'mol0',
  'mol1

In [9]:
ACTIVITY_LABELS = ["anti_cancer_1", "anti_cancer_2", "anti_cancer_3"]

active_molecules = {
    label: set(train.loc[train[label] == 1, "name"]) for label in ACTIVITY_LABELS
}

In [10]:
active_molecules

{'anti_cancer_1': {'mol31',
  'mol32',
  'mol33',
  'mol34',
  'mol35',
  'mol36',
  'mol37',
  'mol38',
  'mol39',
  'mol40',
  'mol41',
  'mol42',
  'mol43',
  'mol44',
  'mol45',
  'mol46'},
 'anti_cancer_2': {'mol13',
  'mol14',
  'mol15',
  'mol16',
  'mol17',
  'mol18',
  'mol19',
  'mol20',
  'mol21',
  'mol22',
  'mol23',
  'mol24',
  'mol25',
  'mol26',
  'mol28',
  'mol29',
  'mol30'},
 'anti_cancer_3': {'mol0',
  'mol1',
  'mol10',
  'mol11',
  'mol12',
  'mol13',
  'mol14',
  'mol15',
  'mol16',
  'mol17',
  'mol18',
  'mol19',
  'mol2',
  'mol20',
  'mol21',
  'mol22',
  'mol23',
  'mol24',
  'mol25',
  'mol26',
  'mol28',
  'mol29',
  'mol3',
  'mol30',
  'mol4',
  'mol5',
  'mol6',
  'mol7',
  'mol8',
  'mol9'}}

In [11]:
counts = {
    "n_molecules": len(train),
    "n_active": {label: len(active_molecules[label]) for label in ACTIVITY_LABELS},
    "n_X": {X: len(mols) for X, mols in molecules_subgraphs_appearance .items()},
    "n_X_and_active": {
        label: {X: len(mols & active_molecules[label]) for X, mols in molecules_subgraphs_appearance.items()}
        for label in ACTIVITY_LABELS
    },
}

In [12]:
counts

{'n_molecules': 46,
 'n_active': {'anti_cancer_1': 16, 'anti_cancer_2': 17, 'anti_cancer_3': 30},
 'n_X': {(1,): 7,
  (2,): 8,
  (3,): 10,
  (4,): 12,
  (5,): 11,
  (6,): 13,
  (7,): 9,
  (8,): 11,
  (9,): 7,
  (10,): 14,
  (11,): 18,
  (12,): 13,
  (13,): 8,
  (14,): 25,
  (15,): 29,
  (16,): 30,
  (17,): 8,
  (18,): 10,
  (19,): 16,
  (20,): 44,
  (21,): 46,
  (1, 2): 1,
  (1, 3): 2,
  (1, 4): 2,
  (1, 5): 5,
  (1, 6): 5,
  (1, 7): 1,
  (1, 8): 2,
  (1, 9): 4,
  (1, 10): 5,
  (1, 11): 5,
  (1, 12): 2,
  (1, 14): 2,
  (1, 15): 2,
  (1, 16): 2,
  (1, 17): 1,
  (1, 19): 2,
  (1, 20): 7,
  (1, 21): 7,
  (2, 3): 6,
  (2, 4): 8,
  (2, 7): 8,
  (2, 8): 6,
  (2, 10): 1,
  (2, 11): 2,
  (2, 12): 8,
  (2, 14): 8,
  (2, 15): 8,
  (2, 16): 8,
  (2, 17): 6,
  (2, 19): 8,
  (2, 20): 8,
  (2, 21): 8,
  (3, 4): 10,
  (3, 7): 6,
  (3, 8): 10,
  (3, 10): 2,
  (3, 11): 4,
  (3, 12): 10,
  (3, 14): 10,
  (3, 15): 10,
  (3, 16): 10,
  (3, 17): 6,
  (3, 19): 10,
  (3, 20): 10,
  (3, 21): 10,
  (4, 7): 8,


In [13]:
def contingency_table(X, label):
    n_molecules = counts["n_molecules"]
    n_X = counts["n_X"][X]
    n_active = counts["n_active"][label]
    n_X_and_active = counts["n_X_and_active"][label][X]

    n_X_active       = n_X_and_active                                 # X present, active
    n_X_inactive     = n_X - n_X_and_active                           # X present, inactive
    n_notX_active    = n_active - n_X_and_active                      # X absent, active
    n_notX_inactive  = n_molecules - n_X - n_active + n_X_and_active  # X absent, inactive
    return n_X_active, n_X_inactive, n_notX_active, n_notX_inactive

In [14]:
def mutual_information(X, label):
    """Mutual Information between X (present/absent) and C = `label` (active/inactive)."""
    n_X_active, n_X_inactive, n_notX_active, n_notX_inactive = contingency_table(X, label)

    n_molecules = counts["n_molecules"]
    n_X = n_X_active + n_X_inactive
    n_active = n_X_active + n_notX_active

    cells = [
        (n_X_active,       n_X,                n_active),
        (n_X_inactive,     n_X,                n_molecules - n_active),
        (n_notX_active,    n_molecules - n_X,  n_active),
        (n_notX_inactive,  n_molecules - n_X,  n_molecules - n_active),
    ]
    mi = 0.0
    for n_cell, n_row, n_col in cells:
        if n_cell > 0:
            mi += n_cell / n_molecules * np.log2(n_cell * n_molecules / (n_row * n_col))
    return mi

def is_positive_rule(X, label):
    """True if X makes C=1 more likely: P(C=1 | X) > P(C=1)."""
    n_X_active, n_X_inactive, n_notX_active, n_notX_inactive = contingency_table(X, label)
    n_X = n_X_active + n_X_inactive
    n_active = n_X_active + n_notX_active
    return n_X_active * counts["n_molecules"] > n_X * n_active


mi_table = pd.DataFrame(
    [(X, label, mutual_information(X, label), is_positive_rule(X, label))
     for label in ACTIVITY_LABELS
     for X in counts["n_X"]],
    columns=["X", "C", "MI", "is_positive"],
)

In [15]:
mi_table

,X,C,MI,is_positive
0,"(1,)",anti_cancer_1,0.002260,False
1,"(2,)",anti_cancer_1,0.318752,True
2,"(3,)",anti_cancer_1,0.423398,True
3,"(4,)",anti_cancer_1,0.545872,True
4,"(5,)",anti_cancer_1,0.175279,False
...,...,...,...,...
559,"(18, 20)",anti_cancer_3,0.156487,True
560,"(18, 21)",anti_cancer_3,0.156487,True
561,"(19, 20)",anti_cancer_3,0.512594,False
562,"(19, 21)",anti_cancer_3,0.512594,False


In [16]:
positive_rules = mi_table[mi_table["is_positive"]]

In [17]:
positive_rules

,X,C,MI,is_positive
1,"(2,)",anti_cancer_1,0.318752,True
2,"(3,)",anti_cancer_1,0.423398,True
3,"(4,)",anti_cancer_1,0.545872,True
6,"(7,)",anti_cancer_1,0.227815,True
7,"(8,)",anti_cancer_1,0.324108,True
...,...,...,...,...
550,"(16, 18)",anti_cancer_3,0.156487,True
554,"(17, 18)",anti_cancer_3,0.027567,True
558,"(18, 19)",anti_cancer_3,0.013591,True
559,"(18, 20)",anti_cancer_3,0.156487,True


In [26]:
def n_and_n_active(X, label):
    """(n(X), n(X and active)). The empty tuple () is the empty set: satisfied by all molecules."""
    if not X:
        return counts["n_molecules"], counts["n_active"][label]
    return counts["n_X"][X], counts["n_X_and_active"][label][X]

def confidence(X, label):
    """P(C=1 | X)."""
    n_X, n_X_active = n_and_n_active(X, label)
    return n_X_active / n_X

def proper_subsets(X):
    return [()] if len(X) == 1 else [(fragment_id,) for fragment_id in X]

def conditional_mutual_information(X, Y, label):
    """I(X; C | Y) in bits, where Y is a subset of X, so X present implies Y present."""
    n_molecules = counts["n_molecules"]
    n_X, n_X_active = n_and_n_active(X, label)
    n_Y, n_Y_active = n_and_n_active(Y, label)

    n_Y_inactive = n_Y - n_Y_active
    cells = [
        (n_X_active,                          n_X,       n_Y_active),    # X present, active
        (n_X - n_X_active,                    n_X,       n_Y_inactive),  # X present, inactive
        (n_Y_active - n_X_active,             n_Y - n_X, n_Y_active),    # Y present, X absent, active
        (n_Y_inactive - (n_X - n_X_active),   n_Y - n_X, n_Y_inactive),  # Y present, X absent, inactive
    ]
    mic = 0.0
    for n_cell, n_group, n_Y_same_activity in cells:
        if n_cell > 0:
            mic += n_cell / n_molecules * np.log2(n_cell * n_Y / (n_group * n_Y_same_activity))
    return mic

rule_table = positive_rules.copy()

rule_table["MI_C"] = [
    min(conditional_mutual_information(X, Y, label) for Y in proper_subsets(X))
    for X, label in zip(rule_table["X"], rule_table["C"])
]
rule_table["improvement"] = [
    int(all(confidence(X, label) > confidence(Y, label) for Y in proper_subsets(X)))
    for X, label in zip(rule_table["X"], rule_table["C"])
]

In [27]:
rule_table

,X,C,MI,is_positive,MI_C,improvement
1,"(2,)",anti_cancer_1,0.318752,True,0.318752,1
2,"(3,)",anti_cancer_1,0.423398,True,0.423398,1
3,"(4,)",anti_cancer_1,0.545872,True,0.545872,1
6,"(7,)",anti_cancer_1,0.227815,True,0.227815,1
7,"(8,)",anti_cancer_1,0.324108,True,0.324108,1
...,...,...,...,...,...,...
550,"(16, 18)",anti_cancer_3,0.156487,True,0.000000,0
554,"(17, 18)",anti_cancer_3,0.027567,True,0.000000,0
558,"(18, 19)",anti_cancer_3,0.013591,True,0.000000,0
559,"(18, 20)",anti_cancer_3,0.156487,True,0.000000,0


In [28]:
MIC_THRESHOLD = 0.02

representative_rules = rule_table[
    (rule_table["improvement"] == 1) & (rule_table["MI_C"] > MIC_THRESHOLD)
].sort_values(["C", "MI"], ascending=[True, False])

In [29]:
len(representative_rules)

48

In [30]:
representative_rules

,X,C,MI,is_positive,MI_C,improvement
3,"(4,)",anti_cancer_1,0.545872,True,0.545872,1
18,"(19,)",anti_cancer_1,0.512594,True,0.512594,1
11,"(12,)",anti_cancer_1,0.439291,True,0.439291,1
2,"(3,)",anti_cancer_1,0.423398,True,0.423398,1
105,"(8, 12)",anti_cancer_1,0.423398,True,0.050680,1
112,"(8, 19)",anti_cancer_1,0.423398,True,0.069288,1
7,"(8,)",anti_cancer_1,0.324108,True,0.324108,1
1,"(2,)",anti_cancer_1,0.318752,True,0.318752,1
95,"(7, 12)",anti_cancer_1,0.318752,True,0.032098,1
96,"(7, 14)",anti_cancer_1,0.318752,True,0.098464,1


In [24]:
test_name = f"mol{test.index[0]}"
test_molecule = Chem.MolFromSmiles(test["SMILES"].values[0].strip())
fragments_in_M = {fragment_id for fragment_id, query in fragment_queries.items() if contains_fragment(test_molecule, query)}

print("M =", test_name, test["SMILES"].values[0])
fragment_table[fragment_table["id"].isin(fragments_in_M)][["id", "description", "s_abs"]]

M = mol27 COc1cc(C2c3cc4c(cc3OC(O)C2C)OCO4)cc(OC)c1OC


,id,description,s_abs
12,13,O(-C)-c1:c(:c:c(:c:c:1-O-C)-C1-C-C-O-c2:c:c3-O...,8
13,14,O(-C)-c1:c(:c:c(:c:c:1-O-C)-C)-O-C,25
14,15,O(-c1:c:c:c:c(:c:1)-C)-C,29
15,16,O(-c1:c:c:c:c:c:1)-C,30
17,18,O1-C-O-c2:c-1:c:c:c:c:2,10
19,20,c1(:c:c:c:c:c:1)-C,44
20,21,c1:c:c:c:c:c:1,46


In [25]:
MEASURES = ["confidence", "MI", "MI_x_confidence", "lift", "leverage"]

def rule_measures(X, label):
    n_molecules = counts["n_molecules"]
    n_X, n_X_active = n_and_n_active(X, label)
    p_active = counts["n_active"][label] / n_molecules
    rule_confidence = n_X_active / n_X
    return pd.Series({
        "confidence": rule_confidence,
        "lift": rule_confidence / p_active,
        "leverage": n_X_active / n_molecules - (n_X / n_molecules) * p_active,
    })

scored_rules = representative_rules[["X", "C", "MI"]].reset_index(drop=True)
scored_rules = pd.concat([scored_rules, scored_rules.apply(lambda r: rule_measures(r.X, r.C), axis=1)], axis=1)
scored_rules["MI_x_confidence"] = scored_rules["MI"] * scored_rules["confidence"]
scored_rules = scored_rules[["X", "C", *MEASURES]]

def rules_matching(fragments):
    """Representative rules X -> C with every substructure of X among `fragments`."""
    return scored_rules[scored_rules["X"].apply(lambda X: set(X) <= fragments)]

rules_for_M = rules_matching(fragments_in_M)
rules_for_M

,X,C,confidence,MI,MI_x_confidence,lift,leverage
18,"(14,)",anti_cancer_1,0.560000,0.187161,0.104810,1.610000,0.115312
19,"(15,)",anti_cancer_1,0.482759,0.109098,0.052668,1.387931,0.085066
20,"(16, 20)",anti_cancer_1,0.482759,0.109098,0.052668,1.387931,0.085066
22,"(16,)",anti_cancer_1,0.466667,0.092964,0.043383,1.341667,0.077505
26,"(20,)",anti_cancer_1,0.363636,0.027567,0.010024,1.045455,0.015123
27,"(18,)",anti_cancer_2,1.000000,0.394156,0.394156,2.705882,0.137051
28,"(13,)",anti_cancer_2,1.000000,0.297936,0.297936,2.705882,0.109641
29,"(15,)",anti_cancer_2,0.517241,0.127324,0.065857,1.399594,0.093100
30,"(16, 20)",anti_cancer_2,0.517241,0.127324,0.065857,1.399594,0.093100
31,"(16,)",anti_cancer_2,0.500000,0.109098,0.054549,1.352941,0.085066


In [26]:
def group_scores(rules):
    scores = rules.groupby("C")[MEASURES].mean().reindex(ACTIVITY_LABELS)
    scores.insert(0, "n_rules", rules.groupby("C").size().reindex(ACTIVITY_LABELS, fill_value=0))
    return scores

scores_M = group_scores(rules_for_M)
scores_M

,n_rules,confidence,MI,MI_x_confidence,lift,leverage
C,,,,,,
anti_cancer_1,5,0.471164,0.105178,0.052711,1.354597,0.075614
anti_cancer_2,6,0.653474,0.180934,0.148309,1.768225,0.089004
anti_cancer_3,2,1.000000,0.138715,0.138715,1.533333,0.068053


In [27]:
fragments_in_train_molecule = {
    name: {X[0] for X, names in molecules_subgraphs_appearance.items() if len(X) == 1 and name in names}
    for name in train["name"]
}

train_scores = pd.concat(
    {name: group_scores(rules_matching(fragments)) for name, fragments in fragments_in_train_molecule.items()},
    names=["name", "C"],
).reset_index()
train_scores["true"] = [
    int(train.loc[train["name"] == name, label].iloc[0])
    for name, label in zip(train_scores["name"], train_scores["C"])
]

def fit_threshold(scores, truth):
    """Threshold t (predict active iff score >= t) with the best accuracy; a missing score counts as inactive."""
    scores = np.nan_to_num(scores, nan=-np.inf)
    values = np.unique(scores[np.isfinite(scores)])
    candidates = np.concatenate([[values[0] - 1e-9], (values[:-1] + values[1:]) / 2, [values[-1] + 1e-9]])
    accuracies = [((scores >= t).astype(int) == truth).mean() for t in candidates]
    best = int(np.argmax(accuracies))
    return candidates[best], accuracies[best]

def predict(score, threshold):
    return int(not np.isnan(score) and score >= threshold)

prediction_rows = []
for label in ACTIVITY_LABELS:
    label_scores = train_scores[train_scores["C"] == label]
    true_M = int(test[label].iloc[0])

    fixed_train_accuracy = (label_scores["confidence"].fillna(0).ge(0.5).astype(int) == label_scores["true"]).mean()
    prediction_rows.append((label, "confidence >= 0.5", 0.5, fixed_train_accuracy,
                            scores_M.loc[label, "confidence"], predict(scores_M.loc[label, "confidence"], 0.5), true_M))

    for measure in MEASURES:
        threshold, train_accuracy = fit_threshold(label_scores[measure].to_numpy(), label_scores["true"].to_numpy())
        score = scores_M.loc[label, measure]
        prediction_rows.append((label, f"{measure} (fitted)", threshold, train_accuracy, score, predict(score, threshold), true_M))

predictions = pd.DataFrame(
    prediction_rows, columns=["C", "decision rule", "threshold", "train accuracy", "score of M", "predicted", "true"]
)
predictions["correct"] = predictions["predicted"] == predictions["true"]
predictions.round(4)

,C,decision rule,threshold,train accuracy,score of M,predicted,true,correct
0,anti_cancer_1,confidence >= 0.5,0.5000,0.8478,0.4712,0,0,True
1,anti_cancer_1,confidence (fitted),0.6064,0.9348,0.4712,0,0,True
2,anti_cancer_1,MI (fitted),0.2330,0.9565,0.1052,0,0,True
3,anti_cancer_1,MI_x_confidence (fitted),0.1943,0.9565,0.0527,0,0,True
4,anti_cancer_1,lift (fitted),1.7433,0.9348,1.3546,0,0,True
5,anti_cancer_1,leverage (fitted),0.0947,0.9348,0.0756,0,0,True
6,anti_cancer_2,confidence >= 0.5,0.5000,0.8261,0.6535,1,1,True
7,anti_cancer_2,confidence (fitted),0.5421,0.8478,0.6535,1,1,True
8,anti_cancer_2,MI (fitted),0.1324,0.8261,0.1809,1,1,True
9,anti_cancer_2,MI_x_confidence (fitted),0.0615,0.8478,0.1483,1,1,True


In [28]:
summary = predictions.pivot(index="decision rule", columns="C", values="predicted")
summary.loc["true label"] = test[ACTIVITY_LABELS].iloc[0]
summary

C,anti_cancer_1,anti_cancer_2,anti_cancer_3
decision rule,,,
MI (fitted),0,1,1
MI_x_confidence (fitted),0,1,1
confidence (fitted),0,1,1
confidence >= 0.5,0,1,1
leverage (fitted),0,1,1
lift (fitted),0,1,1
true label,0,1,1


In [29]:
train_scores.assign(no_rule=train_scores["n_rules"] == 0).groupby(["C", "true"])["no_rule"].agg(["sum", "size"])

sum  size
C             true           
anti_cancer_1 0       1    30
              1       0    16
anti_cancer_2 0       1    29
              1       0    17
anti_cancer_3 0      12    16
              1       7    30